In [1]:
from transformers import AutoTokenizer, AutoModel

In [2]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
text = "..." * 600

inputs = tokenizer(text, return_tensors="pt", truncation=True)
outputs = model(**inputs)

In [6]:
print("\n" + "="*60)
print("📝 EXERCISE FOR YOU (Chapter 8 - Long Sequences):")
print("="*60)
print("1. Based on what you learned in Chapter 8, what causes this error?")
print("2. How can you fix it? (Hint: Think about the 'truncation'")
print("   parameter in the tokenizer)")


📝 EXERCISE FOR YOU (Chapter 8 - Long Sequences):
1. Based on what you learned in Chapter 8, what causes this error?
2. How can you fix it? (Hint: Think about the 'truncation'
   parameter in the tokenizer)


In [7]:
print("\n" + "="*60)
print("✅ SOLUTION:")
print("="*60)


✅ SOLUTION:


In [8]:
print("\n🔧 Solution 1: Use truncation")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
outputs = model(**inputs)

print("✅ Success! The text was truncated to 512 tokens.")
print(f"   Input shape: {inputs['input_ids'].shape}")


🔧 Solution 1: Use truncation


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Success! The text was truncated to 512 tokens.
   Input shape: torch.Size([1, 512])


In [28]:
print("\n🔧 Solution 2: Sliding window (alternative approach)")
def process_long_text(text, tokenizer, model, max_length=512, stride=256):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True
    )
    all_outputs = []
    for i in range(len(inputs['input_ids'])):
        window_inputs = {
            'input_ids': inputs['input_ids'][i].unsqueeze(0),
            'attention_mask': inputs['attention_mask'][i].unsqueeze(0)
        }
        output = model(**window_inputs)
        all_outputs.append(output)
    return all_outputs


🔧 Solution 2: Sliding window (alternative approach)


In [29]:
text = "This is a long text. " * 200  # ~1000 words
results = process_long_text(text, tokenizer, model)
print(f"✅ Processed {len(results)} windows of 512 tokens each")
print(f"   Window 1 shape: {results[0].last_hidden_state.shape}")

✅ Processed 4 windows of 512 tokens each
   Window 1 shape: torch.Size([1, 512, 768])


In [30]:
print("\n" + "="*60)
print("📚 CHAPTER 8: UNDERSTANDING THE ERROR")
print("="*60)


📚 CHAPTER 8: UNDERSTANDING THE ERROR


In [32]:
print("\n" + "="*60)
print("📊 QUICK REFERENCE: MODEL MAX LENGTHS")
print("="*60)
print("""
| Model          | Max Length | Tokens for 600 words |
|----------------|------------|----------------------|
| BERT           | 512        | ~800+ (ERROR!)       |
| GPT-2          | 1024       | ~800 (OK)            |
| DistilBERT     | 512        | ~800+ (ERROR!)       |
| RoBERTa        | 512        | ~800+ (ERROR!)       |
| Longformer     | 4096       | ~800 (OK)            |
| BigBird        | 4096       | ~800 (OK)            |
""")


📊 QUICK REFERENCE: MODEL MAX LENGTHS

| Model          | Max Length | Tokens for 600 words |
|----------------|------------|----------------------|
| BERT           | 512        | ~800+ (ERROR!)       |
| GPT-2          | 1024       | ~800 (OK)            |
| DistilBERT     | 512        | ~800+ (ERROR!)       |
| RoBERTa        | 512        | ~800+ (ERROR!)       |
| Longformer     | 4096       | ~800 (OK)            |
| BigBird        | 4096       | ~800 (OK)            |



In [33]:
print("\n" + "="*60)
print("✅ EXERCISE COMPLETE!")
print("="*60)
print("\n📝 Answers to your exercise:")
print("1. The error occurs because the input exceeds BERT's maximum")
print("   sequence length of 512 tokens.")
print("\n2. Fix it by adding:")
print("   truncation=True and max_length=512 to the tokenizer")
print("\n🎯 Bonus: For even longer texts, use the sliding window")
print("   approach with return_overflowing_tokens=True")


✅ EXERCISE COMPLETE!

📝 Answers to your exercise:
1. The error occurs because the input exceeds BERT's maximum
   sequence length of 512 tokens.

2. Fix it by adding:
   truncation=True and max_length=512 to the tokenizer

🎯 Bonus: For even longer texts, use the sliding window
   approach with return_overflowing_tokens=True
